# 05 — Weight Sensitivity Analysis

## tl;dr

When delivery risk is increased to 60% of the model, the full supplier ranking retains a **0.91 Spearman correlation** with the baseline, but **10 suppliers change risk tiers**. The overall ordering is fairly stable, while threshold-based actions for borderline suppliers remain sensitive to management priorities.

## Context & Methods

This notebook asks: **Would management reach similar priorities under a different but plausible set of risk weights?** We compare the baseline model with a delivery-focused scenario.

### Key Assumptions

- Scenario inputs are normalized to 100%.
- Spearman correlation measures ordering similarity, not model accuracy.
- Tier changes identify decisions that require additional judgment.
- One scenario cannot establish universal robustness.

### 1. Load baseline scores and define a scenario

In [1]:
from pathlib import Path
import pandas as pd

from src.supply_chain_risk.analytics import RISK_WEIGHTS, apply_weight_scenario, build_analysis, load_data

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
tables, supplier_scores, part_risk, kpis = build_analysis(DATA_DIR)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

scenario_weights = {'delivery_risk': 60, 'quality_risk': 10, 'lead_time_variability_risk': 10, 'geographic_risk': 10, 'financial_risk': 5, 'dependency_risk': 5}
scenario_weights

{'delivery_risk': 60, 'quality_risk': 10, 'lead_time_variability_risk': 10, 'geographic_risk': 10, 'financial_risk': 5, 'dependency_risk': 5}

### 2. Recalculate scores and measure stability

In [2]:
scenario = apply_weight_scenario(supplier_scores, scenario_weights)
baseline_top10 = set(supplier_scores.nlargest(10, "risk_score")["supplier_id"])
scenario_top10 = set(scenario.nsmallest(10, "scenario_rank")["supplier_id"])
pd.DataFrame([
    {"measure": "Top-10 suppliers retained", "value": f"{len(baseline_top10 & scenario_top10)}/10"},
    {"measure": "Spearman rank correlation", "value": f"{scenario[['baseline_rank', 'scenario_rank']].corr(method='spearman').iloc[0, 1]:.3f}"},
    {"measure": "Suppliers changing risk tier", "value": int(scenario["tier_changed"].sum())},
])

,measure,value
0,Top-10 suppliers retained,7/10
1,Spearman rank correlation,0.911
2,Suppliers changing risk tier,10


### 3. Inspect suppliers most affected by the assumption change

In [3]:
movers = scenario.assign(abs_rank_change=scenario["rank_change"].abs()).nlargest(12, "abs_rank_change")
movers[["supplier_name", "baseline_rank", "scenario_rank", "rank_change", "risk_score", "scenario_score", "risk_tier", "scenario_risk_tier"]].round(1)

,supplier_name,baseline_rank,scenario_rank,rank_change,risk_score,scenario_score,risk_tier,scenario_risk_tier
10,Pioneer Materials 33,24,11,13,24.10,28.80,Low,Medium
16,Vector Precision 17,6,17,-11,34.20,24.80,Medium,Low
4,Nova Materials 21,14,5,9,28.80,30.90,Medium,Medium
8,Vector Dynamics 01,18,9,9,27.70,29.10,Medium,Medium
14,Atlas Precision 07,7,15,-8,32.40,25.20,Medium,Medium
23,Pioneer Dynamics 30,17,24,-7,27.90,21.90,Medium,Low
26,Summit Dynamics 27,20,27,-7,26.90,18.60,Medium,Low
37,Pioneer Components 09,31,38,-7,18.80,11.40,Low,Low
13,Nova Materials 32,8,14,-6,32.30,25.70,Medium,Medium
19,Nova Dynamics 15,26,20,6,22.70,23.80,Low,Low


### 4. Identify threshold-sensitive suppliers

In [4]:
scenario.loc[scenario["tier_changed"], ["supplier_name", "risk_score", "scenario_score", "risk_tier", "scenario_risk_tier"]].sort_values("scenario_score", ascending=False)

,supplier_name,risk_score,scenario_score,risk_tier,scenario_risk_tier
2,Aero Components 04,39.50,33.00,High,Medium
6,Vector Systems 34,36.30,30.50,High,Medium
10,Pioneer Materials 33,24.10,28.80,Low,Medium
15,Vector Precision 12,31.90,25.00,Medium,Low
16,Vector Precision 17,34.20,24.80,Medium,Low
17,Vector Dynamics 39,28.80,24.80,Medium,Low
18,Aero Systems 35,26.80,24.30,Medium,Low
21,Pioneer Dynamics 23,27.20,23.10,Medium,Low
23,Pioneer Dynamics 30,27.90,21.90,Medium,Low
26,Summit Dynamics 27,26.90,18.60,Medium,Low


## Results

The delivery-focused scenario preserves most of the overall ordering, but several suppliers cross portfolio risk thresholds. Those threshold-sensitive suppliers should be reviewed with additional evidence rather than treated as automatic escalations or removals.

## Takeaways

- Ranking stability and tier stability are different questions.
- Sensitivity analysis exposes where subjective assumptions affect decisions.
- Robust suppliers remain priorities across scenarios.
- Borderline suppliers require more evidence, calibrated thresholds, and business-owner review.